In [14]:
import traci
import os
import sys
import random

# Close previous connection
if traci.isLoaded():
    traci.close()

# SUMO setup
if 'SUMO_HOME' in os.environ:
    tools = os.path.join(os.environ['SUMO_HOME'], 'tools')
    sys.path.append(tools)
else:
    raise Exception("SUMO_HOME not set")

sumoBinary = "sumo-gui"
sumoCmd = [sumoBinary, "-c", "../sumo_files/simple.sumocfg"]

traci.start(sumoCmd)

# Get one traffic light
tls = traci.trafficlight.getIDList()[0]
print("Using TLS:", tls)

# ---------------- STATE ----------------
def get_state(tls_id):
    lanes = traci.trafficlight.getControlledLanes(tls_id)

    total = 0
    for lane in lanes:
        total += traci.lane.getLastStepVehicleNumber(lane)

    return int(total)

# ---------------- REWARD ----------------
def get_reward(state):
    return -state

# ---------------- Q-TABLE ----------------
Q={}

# ---------------- PARAMETERS ----------------
alpha=0.1
gamma=0.9
epsilon =0.2

# ---------------- ACTION ----------------
def choose_action(state):
    if(random.random() < epsilon):
        return random.choice([0,1])
    if(state,0) not in Q:
        Q[(state,0)]=0
    if(state,1) not in Q:
        Q[(state,1)]=0
    return 0 if Q[(state,0)] > Q[(state,1)] else 1
# ---------------- SIMULATION LOOP ----------------
current_phase=0

for step in range(1000):
    traci.simulationStep()

    state=get_state(tls)
    action=choose_action(state)

    if action==1:
        current_phase=2 if current_phase==0 else 0
        traci.trafficlight.setPhase(tls,current_phase)
    reward=get_reward(state)
    next_state=get_state(tls)

    if(state,action) not in Q:
        Q[(state,action)]=0
    if(next_state,0) not in Q:
        Q[(next_state,0)]=0
    if(next_state,1) not in Q:
        Q[(next_state,1)]=0

    Q[(state,action)]=Q[(state,action)]+alpha*(reward + gamma*max(Q[(next_state,1)],Q[(next_state,1)])-Q[(state,action)])
    print(f"Step: {step} | State: {state} | Action: {action} | Reward: {reward}")
traci.close()

Using TLS: A1
Step: 0 | State: 0 | Action: 1 | Reward: 0
Step: 1 | State: 0 | Action: 1 | Reward: 0
Step: 2 | State: 0 | Action: 1 | Reward: 0
Step: 3 | State: 0 | Action: 1 | Reward: 0
Step: 4 | State: 3 | Action: 1 | Reward: -3
Step: 5 | State: 6 | Action: 1 | Reward: -6
Step: 6 | State: 9 | Action: 1 | Reward: -9
Step: 7 | State: 9 | Action: 0 | Reward: -9
Step: 8 | State: 12 | Action: 1 | Reward: -12
Step: 9 | State: 12 | Action: 0 | Reward: -12
Step: 10 | State: 12 | Action: 1 | Reward: -12
Step: 11 | State: 12 | Action: 0 | Reward: -12
Step: 12 | State: 12 | Action: 1 | Reward: -12
Step: 13 | State: 12 | Action: 0 | Reward: -12
Step: 14 | State: 12 | Action: 1 | Reward: -12
Step: 15 | State: 12 | Action: 0 | Reward: -12
Step: 16 | State: 6 | Action: 0 | Reward: -6
Step: 17 | State: 6 | Action: 1 | Reward: -6
Step: 18 | State: 3 | Action: 0 | Reward: -3
Step: 19 | State: 3 | Action: 1 | Reward: -3
Step: 20 | State: 3 | Action: 0 | Reward: -3
Step: 21 | State: 6 | Action: 0 | Rewar